In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;

# Module 2: 데이터 탐색 및 전처리

이 노트북에서는 TPC-H 데이터를 활용하여 고객 LTV 예측 모델을 위한 **탐색적 데이터 분석(EDA)** 및 **전처리 파이프라인**을 구축합니다.

**데이터 소스:** `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1`

**목표 변수:** `FUTURE_12M_REVENUE` — 고객의 향후 12개월 예상 구매금액 (연속형, 단위: $)

---
### 목차
1. Setup — 라이브러리 임포트 및 Snowpark 세션 초기화
2. Schema 생성
3. 데이터 탐색 (EDA)
4. 타겟 변수 생성 및 피처 테이블 저장
5. 결측값 및 이상치 처리
6. 전처리 파이프라인 (Snowflake ML Preprocessors)

## 1. Setup

필요한 라이브러리를 임포트하고 Snowpark 세션을 초기화합니다.
Snowflake Notebooks Container Runtime에서는 `get_active_session()`으로 세션을 가져옵니다.

In [ ]:
# ── 표준 라이브러리 ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
# ── 데이터 처리 ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
# ── 시각화 (Snowflake Workspaces: matplotlib만 지원) ─────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
# ── 한글 폰트 설정 (Snowflake Workspaces Container Runtime) ─────────────
import subprocess, matplotlib, matplotlib.font_manager as fm
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'],
               capture_output=True)  # NanumGothic 설치
fm._load_fontmanager(try_read_cache=False)
nanum = [f for f in fm.findSystemFonts() if 'Nanum' in f]
if nanum:
    fm.fontManager.addfont(nanum[0])
    matplotlib.rc('font', family='NanumGothic')
else:
    # 폰트 없으면 fallback: 그래프 제목/레이블을 영어로 표시
    pass
matplotlib.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지
# ── Snowpark ─────────────────────────────────────────────────────────────────
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import DoubleType, LongType, StringType
# ── Snowflake ML ─────────────────────────────────────────────────────────────
from snowflake.ml.modeling.preprocessing import StandardScaler, OrdinalEncoder
from snowflake.ml.modeling.pipeline import Pipeline
# ── 세션 초기화 ───────────────────────────────────────────────────────────────
session = get_active_session()
print('세션 초기화 완료')

## 2. Schema 생성

`DEMO.ML_DEMO` 스키마가 없으면 생성합니다. 이미 존재할 경우 건너뜁니다.

In [ ]:
# DEMO.ML_DEMO 스키마 생성 (없을 경우)
session.sql("CREATE DATABASE IF NOT EXISTS DEMO").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS DEMO.ML_DEMO").collect()

print("✅ DEMO.ML_DEMO 스키마 준비 완료")

## 3. 데이터 탐색 (EDA)

TPC-H 데이터셋의 세 가지 주요 테이블을 탐색합니다:
- **CUSTOMER**: 고객 기본 정보 (시장 세그먼트, 계좌 잔액 등)
- **ORDERS**: 주문 정보 (주문 날짜, 금액, 상태 등)
- **LINEITEM**: 주문 라인 아이템 (수량, 가격, 할인율 등)

모든 데이터는 `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1`에서 가져옵니다.

In [ ]:
# ── TPC-H 테이블 로드 ─────────────────────────────────────────────────────────
df_customer = session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER")
df_orders   = session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS")
df_lineitem = session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM")

print("데이터 로드 완료 (Snowpark lazy evaluation — 아직 데이터 이동 없음)")

In [ ]:
# ── 행 수 확인 ────────────────────────────────────────────────────────────────
counts = {
    "CUSTOMER": df_customer.count(),
    "ORDERS":   df_orders.count(),
    "LINEITEM": df_lineitem.count(),
}
print("테이블별 행 수:")
for tbl, cnt in counts.items():
    print(f"  {tbl:10s}: {cnt:>12,}")

In [ ]:
# ── 스키마 (컬럼 정보) 출력 ───────────────────────────────────────────────────
print("=== CUSTOMER 스키마 ===")
df_customer.schema.json()
for field in df_customer.schema.fields:
    print(f"  {field.name:20s}  {str(field.datatype)}")

print("\n=== ORDERS 스키마 ===")
for field in df_orders.schema.fields:
    print(f"  {field.name:20s}  {str(field.datatype)}")

print("\n=== LINEITEM 스키마 ===")
for field in df_lineitem.schema.fields:
    print(f"  {field.name:20s}  {str(field.datatype)}")

In [ ]:
# ── 샘플 행 확인 ──────────────────────────────────────────────────────────────
print("=== CUSTOMER 샘플 (5행) ===")
df_customer.show(5)

print("\n=== ORDERS 샘플 (5행) ===")
df_orders.show(5)

print("\n=== LINEITEM 샘플 (5행) ===")
df_lineitem.show(5)

In [ ]:
# ── 기술 통계 ─────────────────────────────────────────────────────────────────
print("=== CUSTOMER 기술 통계 ===")
customer_pd = df_customer.select(
    "C_CUSTKEY", "C_ACCTBAL", "C_NATIONKEY"
).to_pandas()
display(customer_pd.describe())

print("\n=== ORDERS 기술 통계 ===")
orders_pd = df_orders.select(
    "O_ORDERKEY", "O_CUSTKEY", "O_TOTALPRICE", "O_SHIPPRIORITY"
).to_pandas()
display(orders_pd.describe())

In [ ]:
# ── C_MKTSEGMENT 분포 시각화 ──────────────────────────────────────────────────
mktseg_dist = (
    df_customer
    .group_by("C_MKTSEGMENT")
    .agg(F.count("*").alias("COUNT"))
    .order_by("COUNT", ascending=False)
    .to_pandas()
)

colors = ["#636EFA","#EF553B","#00CC96","#AB63FA","#FFA15A"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(mktseg_dist["C_MKTSEGMENT"], mktseg_dist["COUNT"],
       color=colors[:len(mktseg_dist)])
ax.set_title("고객 시장 세그먼트 분포 (C_MKTSEGMENT)", fontsize=14)
ax.set_xlabel("시장 세그먼트")
ax.set_ylabel("고객 수")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

print("시장 세그먼트별 고객 수:")
display(mktseg_dist)


In [ ]:
# ── C_ACCTBAL 분포 시각화 ─────────────────────────────────────────────────────
acctbal_pd = df_customer.select("C_ACCTBAL").to_pandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(acctbal_pd["C_ACCTBAL"].dropna(), bins=50, color="#1f77b4", edgecolor="white")
ax.set_title("고객 계좌 잔액 분포 (C_ACCTBAL)", fontsize=14)
ax.set_xlabel("계좌 잔액")
ax.set_ylabel("고객 수")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

print("C_ACCTBAL 통계:")
print(acctbal_pd["C_ACCTBAL"].describe().to_string())


In [ ]:
# ── 주문 상태 분포 시각화 ─────────────────────────────────────────────────────
order_status_dist = (
    df_orders
    .group_by("O_ORDERSTATUS")
    .agg(F.count("*").alias("COUNT"))
    .order_by("COUNT", ascending=False)
    .to_pandas()
)

status_labels = {"F": "완료(F)", "O": "진행중(O)", "P": "보류(P)"}
order_status_dist["STATUS_LABEL"] = order_status_dist["O_ORDERSTATUS"].map(
    lambda x: status_labels.get(x, x)
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    order_status_dist["COUNT"],
    labels=order_status_dist["STATUS_LABEL"],
    autopct="%1.1f%%",
    colors=["#AEC6CF","#FFD1DC","#B5EAD7"],
    startangle=90,
)
ax.set_title("주문 상태 분포 (O_ORDERSTATUS)", fontsize=14)
plt.tight_layout()
plt.show()

print("주문 상태별 건수:")
display(order_status_dist)


In [ ]:
# ── 주문 금액 분포 ────────────────────────────────────────────────────────────
totalprice_pd = df_orders.select("O_TOTALPRICE").sample(frac=0.05).to_pandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(totalprice_pd["O_TOTALPRICE"].dropna(), bins=60,
        color="#ff7f0e", edgecolor="white")
ax.set_title("주문 금액 분포 — 5% 샘플 (O_TOTALPRICE)", fontsize=14)
ax.set_xlabel("주문 총액")
ax.set_ylabel("주문 수")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()


## 4. 타겟 변수 생성

세 테이블을 조인하여 **고객 단위 집계 피처**를 생성하고 `FUTURE_12M_REVENUE` 이진 레이블을 부여합니다.

| 피처 | 설명 |
|---|---|
| `C_ACCTBAL` | 고객 계좌 잔액 |
| `C_MKTSEGMENT` | 시장 세그먼트 (범주형) |
| `TOTAL_ORDERS` | 총 주문 건수 |
| `AVG_ORDER_VALUE` | 평균 주문 금액 |
| `TOTAL_REVENUE` | 총 매출액 (단가 × 수량 × (1 - 할인율)) |
| `AVG_DISCOUNT` | 평균 할인율 |
| `AVG_QUANTITY` | 평균 주문 수량 |
| `FUTURE_12M_REVENUE` | **타겟**: 1997.07~1998.06 실제 구매금액 ($), 미구매 시 0 |

In [ ]:
# ── Step 1: ORDERS + LINEITEM 집계 (관찰기간만) ──────────────────────────────
# ⚠️  반드시 관찰기간(1992.01.01 ~ 1997.06.30) 데이터만 사용합니다.
OBSERVATION_START = "1992-01-01"
OBSERVATION_END   = "1997-06-30"

df_orders_obs = df_orders.filter(
    (F.col("O_ORDERDATE") >= F.lit(OBSERVATION_START)) &
    (F.col("O_ORDERDATE") <= F.lit(OBSERVATION_END))
)
print(f"관찰기간 주문 수: {df_orders_obs.count():,} (전체: {df_orders.count():,})")

# LINEITEM 집계 (주문 단위)
lineitem_agg = (
    df_lineitem
    .group_by("L_ORDERKEY")
    .agg(
        F.sum(
            F.col("L_EXTENDEDPRICE") * (F.lit(1) - F.col("L_DISCOUNT"))
        ).alias("LINE_REVENUE"),
        F.avg("L_DISCOUNT").alias("AVG_DISCOUNT"),
        F.avg("L_QUANTITY").alias("AVG_QUANTITY"),
    )
)

# 관찰기간 ORDERS + LINEITEM 조인
orders_with_line = (
    df_orders_obs
    .join(
        lineitem_agg,
        df_orders_obs["O_ORDERKEY"] == lineitem_agg["L_ORDERKEY"],
        how="inner",
    )
    .select(
        df_orders_obs["O_CUSTKEY"],
        df_orders_obs["O_TOTALPRICE"],
        df_orders_obs["O_ORDERDATE"],   # LAST_ORDER_DATE 계산용
        lineitem_agg["LINE_REVENUE"],
        lineitem_agg["AVG_DISCOUNT"],
        lineitem_agg["AVG_QUANTITY"],
    )
)

# 고객별 집계
customer_orders_agg = (
    orders_with_line
    .group_by("O_CUSTKEY")
    .agg(
        F.count("*").alias("TOTAL_ORDERS"),
        F.avg("O_TOTALPRICE").alias("AVG_ORDER_VALUE"),
        F.sum("LINE_REVENUE").alias("TOTAL_REVENUE"),
        F.avg("AVG_DISCOUNT").alias("AVG_DISCOUNT"),
        F.avg("AVG_QUANTITY").alias("AVG_QUANTITY"),
        F.max("O_ORDERDATE").alias("LAST_ORDER_DATE"),  # 마지막 구매일
    )
)

print("\nORDERS + LINEITEM 집계 완료 (관찰기간만)")
customer_orders_agg.show(5)


In [ ]:
# ── Step 2: CUSTOMER 조인 + DAYS_SINCE_LAST_ORDER 계산 ───────────────────────
customer_features = (
    df_customer
    .join(
        customer_orders_agg,
        df_customer["C_CUSTKEY"] == customer_orders_agg["O_CUSTKEY"],
        how="inner",
    )
    .select(
        df_customer["C_CUSTKEY"],
        df_customer["C_MKTSEGMENT"],
        df_customer["C_ACCTBAL"],
        df_customer["C_NATIONKEY"],
        customer_orders_agg["TOTAL_ORDERS"],
        customer_orders_agg["AVG_ORDER_VALUE"],
        customer_orders_agg["TOTAL_REVENUE"],
        customer_orders_agg["AVG_DISCOUNT"],
        customer_orders_agg["AVG_QUANTITY"],
        # 관찰기간 마지막 주문 후 경과일 (1997-06-30 기준)
        F.datediff(
            "day",
            customer_orders_agg["LAST_ORDER_DATE"],
            F.to_date(F.lit("1997-06-30"))
        ).alias("DAYS_SINCE_LAST_ORDER"),
    )
)

print(f"조인 후 고객 수: {customer_features.count():,}")
customer_features.show(5)


In [ ]:
# ── Step 3: FUTURE_12M_REVENUE 레이블 생성 ─────────────────────────────────────────
# 예측 윈도우(1997.07.01 ~ 1998.06.30)의 실제 구매금액을 레이블로 사용합니다.
# ⚠️  이 기간 데이터는 레이블 생성에만 사용합니다. 피처로 절대 사용하지 않습니다.

future_revenue = (
    session.table("SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS")
    .filter(
        (F.col("O_ORDERDATE") >= F.lit("1997-07-01")) &
        (F.col("O_ORDERDATE") <= F.lit("1998-06-30"))
    )
    .group_by("O_CUSTKEY")
    .agg(F.sum("O_TOTALPRICE").alias("FUTURE_12M_REVENUE"))
)

# 관찰기간 피처 + 예측 윈도우 레이블 JOIN
# LEFT JOIN: 예측 윈도우에 구매 없는 고객 → FUTURE_12M_REVENUE = $0
customer_features_labeled = (
    customer_features
    .join(
        future_revenue,
        customer_features["C_CUSTKEY"] == future_revenue["O_CUSTKEY"],
        join_type="left"
    )
    .with_column(
        "FUTURE_12M_REVENUE",
        F.coalesce(F.col("FUTURE_12M_REVENUE"), F.lit(0.0))
    )
    .drop("O_CUSTKEY")
)

print("FUTURE_12M_REVENUE 레이블이 추가된 샘플:")
customer_features_labeled.select(
    "C_CUSTKEY", "TOTAL_ORDERS", "AVG_ORDER_VALUE", "DAYS_SINCE_LAST_ORDER", "FUTURE_12M_REVENUE"
).show(10)

# LTV 분포 요약
ltv_stats = customer_features_labeled.select("FUTURE_12M_REVENUE").to_pandas()
zero_cnt = (ltv_stats["FUTURE_12M_REVENUE"] == 0).sum()
pos_cnt  = (ltv_stats["FUTURE_12M_REVENUE"] > 0).sum()
print(f"\nLTV 분포:")
print(f"  $0     (미구매): {zero_cnt:>7,} ({zero_cnt/len(ltv_stats)*100:.1f}%)")
print(f"  $0 초과 (구매): {pos_cnt:>7,} ({pos_cnt/len(ltv_stats)*100:.1f}%)")
print(f"  평균 LTV: ${ltv_stats['FUTURE_12M_REVENUE'].mean():>12,.0f}")
print(f"  중앙값 LTV: ${ltv_stats['FUTURE_12M_REVENUE'].median():>10,.0f}")


In [ ]:
# ── Step 4: CUSTOMER_FEATURES 테이블로 저장 ───────────────────────────────────
(
    customer_features_labeled
    .write
    .mode("overwrite")
    .save_as_table("DEMO.ML_DEMO.CUSTOMER_FEATURES")
)

# 저장된 테이블 확인
saved_df = session.table("DEMO.ML_DEMO.CUSTOMER_FEATURES")
print(f"✅ DEMO.ML_DEMO.CUSTOMER_FEATURES 저장 완료 — 행 수: {saved_df.count():,}")
saved_df.show(5)

## 5. 결측값 및 이상치 처리

모델 학습 전, 데이터 품질을 확인합니다:
- **결측값(Null)** 체크
- **C_ACCTBAL 이상치** 탐지 (IQR 방법)
- **LTV 분포** 확인

In [ ]:
# ── 결측값 확인 ───────────────────────────────────────────────────────────────
df_feat = session.table("DEMO.ML_DEMO.CUSTOMER_FEATURES")
total_rows = df_feat.count()

print(f"전체 행 수: {total_rows:,}\n")
print("컬럼별 결측값(NULL) 수:")

numeric_cols = [
    "C_ACCTBAL", "TOTAL_ORDERS", "AVG_ORDER_VALUE",
    "TOTAL_REVENUE", "AVG_DISCOUNT", "AVG_QUANTITY",
]
cat_cols = ["C_MKTSEGMENT"]

null_counts = df_feat.select(
    *[
        F.sum(F.when(F.col(c).is_null(), F.lit(1)).otherwise(F.lit(0))).alias(c)
        for c in numeric_cols + cat_cols
    ]
).to_pandas()

null_series = null_counts.iloc[0]
for col_name, null_cnt in null_series.items():
    pct = null_cnt / total_rows * 100
    status = "⚠️" if null_cnt > 0 else "✅"
    print(f"  {status} {col_name:25s}: {null_cnt:>6,} ({pct:.2f}%)")

In [ ]:
# ── C_ACCTBAL 이상치 탐지 (IQR 방법) ─────────────────────────────────────────
quantiles = (
    df_feat
    .select(
        F.percentile_cont(0.25).within_group(F.col("C_ACCTBAL")).alias("Q1"),
        F.percentile_cont(0.75).within_group(F.col("C_ACCTBAL")).alias("Q3"),
    )
    .collect()[0]
)
q1 = float(quantiles["Q1"])
q3 = float(quantiles["Q3"])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
print(f"C_ACCTBAL IQR 분석:")
print(f"  Q1 (25%): {q1:>10,.2f}")
print(f"  Q3 (75%): {q3:>10,.2f}")
print(f"  IQR:      {iqr:>10,.2f}")
print(f"  하한 경계: {lower_bound:>10,.2f}")
print(f"  상한 경계: {upper_bound:>10,.2f}")
outlier_count = df_feat.filter(
    (F.col("C_ACCTBAL") < F.lit(lower_bound)) |
    (F.col("C_ACCTBAL") > F.lit(upper_bound))
).count()
pct_outlier = outlier_count / total_rows * 100
print(f"\n  이상치 수: {outlier_count:,} ({pct_outlier:.2f}%)")
# Box plot (matplotlib)
acctbal_sample = df_feat.select("C_ACCTBAL").sample(frac=0.1).to_pandas()
fig, ax = plt.subplots(figsize=(5, 6))
ax.boxplot(acctbal_sample["C_ACCTBAL"].dropna(), vert=True,
           patch_artist=True,
           boxprops=dict(facecolor="#AEC6CF"),
           medianprops=dict(color="black", linewidth=2))
ax.axhline(upper_bound, color="red",  linestyle="--", label=f"IQR 상한 ({upper_bound:,.0f})")
ax.axhline(lower_bound, color="blue", linestyle="--", label=f"IQR 하한 ({lower_bound:,.0f})")
ax.set_title("C_ACCTBAL 박스플롯 (10% 샘플)", fontsize=14)
ax.set_ylabel("계좌 잔액")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── FUTURE_12M_REVENUE (LTV) 분포 확인 ──────────────────────────────────────
ltv_pd = df_feat.select("FUTURE_12M_REVENUE").to_pandas()

# 전체 분포
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 왼쪽: 전체 분포
axes[0].hist(ltv_pd["FUTURE_12M_REVENUE"].dropna(), bins=60,
             color="#636EFA", edgecolor="white")
axes[0].set_title("FUTURE_12M_REVENUE 전체 분포", fontsize=12)
axes[0].set_xlabel("LTV ($)")
axes[0].set_ylabel("고객 수")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# 오른쪽: $0 제외한 양수 분포
positive_ltv = ltv_pd[ltv_pd["FUTURE_12M_REVENUE"] > 0]["FUTURE_12M_REVENUE"]
axes[1].hist(positive_ltv, bins=60, color="#00CC96", edgecolor="white")
axes[1].set_title("FUTURE_12M_REVENUE > $0 분포", fontsize=12)
axes[1].set_xlabel("LTV ($)")
axes[1].set_ylabel("고객 수")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.show()

zero_cnt = (ltv_pd["FUTURE_12M_REVENUE"] == 0).sum()
pos_cnt  = (ltv_pd["FUTURE_12M_REVENUE"] > 0).sum()
print(f"LTV = $0    (이탈): {zero_cnt:>7,} ({zero_cnt/len(ltv_pd)*100:.1f}%)")
print(f"LTV > $0    (유지): {pos_cnt:>7,} ({pos_cnt/len(ltv_pd)*100:.1f}%)")
print(f"평균 LTV: ${ltv_pd['FUTURE_12M_REVENUE'].mean():>12,.0f}")
print(f"중앙값 LTV: ${ltv_pd['FUTURE_12M_REVENUE'].median():>10,.0f}")
print(f"최대 LTV: ${ltv_pd['FUTURE_12M_REVENUE'].max():>12,.0f}")


## 6. 전처리 파이프라인 (Snowflake ML Preprocessors)

Snowflake ML의 `snowflake.ml.modeling.preprocessing` 모듈을 사용하여 전처리를 수행합니다.

- **StandardScaler**: 수치형 피처를 평균 0, 표준편차 1로 정규화
- **OrdinalEncoder**: 범주형 피처(`C_MKTSEGMENT`)를 정수 인코딩

전처리는 Snowflake 내부에서 실행되므로 데이터를 로컬로 이동할 필요가 없습니다.

In [ ]:
# ── 피처 컬럼 정의 ────────────────────────────────────────────────────────────
NUMERIC_FEATURES = [
    "C_ACCTBAL",
    "TOTAL_ORDERS",
    "AVG_ORDER_VALUE",
    "TOTAL_REVENUE",
    "AVG_DISCOUNT",
    "AVG_QUANTITY",
    "DAYS_SINCE_LAST_ORDER",
    "C_NATIONKEY",  # NUMBER 타입이므로 수치형으로 처리
]
CATEGORICAL_FEATURES = ["C_MKTSEGMENT"]
TARGET = "FUTURE_12M_REVENUE"
ID_COL = "C_CUSTKEY"
# 출력 컬럼명 (scaled / encoded)
NUMERIC_OUT   = [f"{c}_SCALED" for c in NUMERIC_FEATURES]
CATEGORICAL_OUT = [f"{c}_ENC" for c in CATEGORICAL_FEATURES]
print("수치형 피처:", NUMERIC_FEATURES)
print("범주형 피처:", CATEGORICAL_FEATURES)
print("타겟:", TARGET)

In [ ]:
# ── OrdinalEncoder: 범주형 인코딩 ────────────────────────────────────────────
encoder = OrdinalEncoder(
    input_cols=CATEGORICAL_FEATURES,
    output_cols=CATEGORICAL_OUT,
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

# fit 후 transform
df_feat = session.table("DEMO.ML_DEMO.CUSTOMER_FEATURES")
encoder.fit(df_feat)
df_encoded = encoder.transform(df_feat)

print("OrdinalEncoder 범주 매핑:")
for col, cats in zip(CATEGORICAL_FEATURES, encoder.categories_):
    for idx, cat in enumerate(cats):
        print(f"  {col}: {cat!r:15s} → {idx}")

print("\n인코딩 결과 샘플:")
df_encoded.select(ID_COL, *CATEGORICAL_FEATURES, *CATEGORICAL_OUT).show(5)

In [ ]:
# ── StandardScaler: 수치형 정규화 ─────────────────────────────────────────────
scaler = StandardScaler(
    input_cols=NUMERIC_FEATURES,
    output_cols=NUMERIC_OUT,
    with_mean=True,
    with_std=True,
)

scaler.fit(df_encoded)
df_transformed = scaler.transform(df_encoded)

print("StandardScaler 적용 완료")
print(f"  정규화 입력 피처: {NUMERIC_FEATURES}")
print(f"  정규화 출력 컬럼: {NUMERIC_OUT}")

# 정규화 전/후 통계를 직접 계산하여 비교
print("\n정규화 전/후 통계 (C_ACCTBAL 예시):")
before = df_encoded.select("C_ACCTBAL").to_pandas()["C_ACCTBAL"]
after  = df_transformed.select("C_ACCTBAL_SCALED").to_pandas()["C_ACCTBAL_SCALED"]
print(f"  원본  — 평균: {before.mean():>12.2f},  표준편차: {before.std():>10.2f}")
print(f"  정규화— 평균: {after.mean():>12.4f},  표준편차: {after.std():>10.4f}")

print("\n변환 결과 샘플 (원본 → 스케일 된 값):")
sample_cols = [ID_COL] + NUMERIC_FEATURES[:3] + NUMERIC_OUT[:3] + [TARGET]
df_transformed.select(sample_cols).show(5)


In [ ]:
# ── 전처리 결과 저장 ──────────────────────────────────────────────────────────
# 모델 학습에 필요한 컬럼만 선택하여 저장
final_cols = [ID_COL] + NUMERIC_OUT + CATEGORICAL_OUT + [TARGET]

(
    df_transformed
    .select(final_cols)
    .write
    .mode("overwrite")
    .save_as_table("DEMO.ML_DEMO.CUSTOMER_FEATURES_PROCESSED")
)

df_final = session.table("DEMO.ML_DEMO.CUSTOMER_FEATURES_PROCESSED")
print(f"✅ DEMO.ML_DEMO.CUSTOMER_FEATURES_PROCESSED 저장 완료")
print(f"   행 수: {df_final.count():,}")
print(f"   컬럼: {df_final.columns}")
df_final.show(5)

In [ ]:
# ── 전처리 전/후 분포 비교 시각화 ─────────────────────────────────────────────
compare_col        = "C_ACCTBAL"
compare_col_scaled = "C_ACCTBAL_SCALED"

orig_sample = df_feat.select(compare_col).sample(frac=0.05).to_pandas()
scaled_sample = df_final.select(compare_col_scaled).sample(frac=0.05).to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("C_ACCTBAL: 정규화 전/후 분포 비교 (5% 샘플)", fontsize=14)

axes[0].hist(orig_sample[compare_col].dropna(), bins=40,
             color="#1f77b4", edgecolor="white")
axes[0].set_title("원본 C_ACCTBAL")
axes[0].set_xlabel("계좌 잔액")
axes[0].set_ylabel("고객 수")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[1].hist(scaled_sample[compare_col_scaled].dropna(), bins=40,
             color="#ff7f0e", edgecolor="white")
axes[1].set_title("정규화 후 C_ACCTBAL_SCALED")
axes[1].set_xlabel("정규화된 값")
axes[1].set_ylabel("고객 수")

plt.tight_layout()
plt.show()

print("\n정규화 후 통계:")
print(scaled_sample[compare_col_scaled].describe().to_string())


## 요약

이 노트북에서 수행한 작업:

| 단계 | 결과물 |
|---|---|
| 데이터 탐색 | CUSTOMER, ORDERS, LINEITEM 테이블 EDA 완료 |
| 피처 생성 | 고객별 15개 집계 피처 + LTV 타겟 변수 생성 |
| 저장 테이블 1 | `DEMO.ML_DEMO.CUSTOMER_FEATURES` (원본 피처 + 레이블) |
| 이상치/결측 처리 | IQR 기반 이상치 확인, 결측값 없음 확인 |
| 전처리 | StandardScaler + OrdinalEncoder 적용 |
| 저장 테이블 2 | `DEMO.ML_DEMO.CUSTOMER_FEATURES_PROCESSED` (모델 입력 준비 완료) |

**다음 단계 (Module 3):** 전처리된 피처를 사용하여 Snowflake ML로 회귀 모델(XGBRegressor/RandomForestRegressor)을 학습합니다.